# 02 - PyTorch 入门

> 配套教程：`04_deep_learning/02_pytorch_basics.py`

PyTorch 是深度学习中最常用的框架之一。它把 NumPy 风格的张量计算、自动求导、GPU 加速、神经网络模块、优化器组合在一起，让我们不必手写反向传播。

## 本 Notebook 概览

| 章节 | 内容 | 关键概念 |
|------|------|----------|
| 1 | Tensor 基础 | 张量创建、形状、矩阵乘法 |
| 2 | Autograd | 计算图、`.backward()`、梯度 |
| 3 | nn.Module | 定义网络结构、参数管理 |
| 4 | 完整训练流程 | forward、loss、zero_grad、backward、step |
| 5 | Sequential 与技巧 | Dropout、BatchNorm、Weight Decay |
| 6 | 保存和加载模型 | `state_dict`、复现实验 |

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager


def configure_chinese_font():
    preferred_fonts = [
        "Noto Sans CJK SC", "Noto Sans CJK JP", "Source Han Sans SC",
        "WenQuanYi Zen Hei", "WenQuanYi Micro Hei", "SimHei",
        "Microsoft YaHei", "PingFang SC", "Arial Unicode MS",
    ]
    available_fonts = {font.name for font in font_manager.fontManager.ttflist}
    for font_name in preferred_fonts:
        if font_name in available_fonts:
            plt.rcParams["font.sans-serif"] = [font_name, "DejaVu Sans"]
            plt.rcParams["font.family"] = "sans-serif"
            print(f"Matplotlib 中文字体: {font_name}")
            return font_name
    print("未检测到可用中文字体，中文图表可能显示为方框。")
    return None


CHINESE_FONT = configure_chinese_font()
plt.rcParams["axes.unicode_minus"] = False
np.set_printoptions(precision=4, suppress=True)
np.random.seed(42)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

torch.manual_seed(42)
print("PyTorch 版本:", torch.__version__)
print("CUDA 可用:", torch.cuda.is_available())

Matplotlib 中文字体: Noto Sans CJK JP
PyTorch 版本: 2.11.0+cpu
CUDA 可用: False


## 1. Tensor 基础

Tensor 可以理解为 PyTorch 里的多维数组，和 NumPy array 很像。但 Tensor 有两个关键能力：

1. 可以放到 GPU 上加速计算。
2. 可以记录计算图并自动求导。

深度学习中的数据、权重、偏置、梯度，本质上都以 Tensor 的形式存在。

In [2]:
t1 = torch.tensor([1.0, 2.0, 3.0])
np_arr = np.array([[1, 2], [3, 4]], dtype=np.float32)
t2 = torch.from_numpy(np_arr)

print("从列表创建:", t1, "dtype=", t1.dtype)
print("从 NumPy 创建:\n", t2)
print("zeros:\n", torch.zeros(2, 3))
print("ones:\n", torch.ones(2, 3))
print("randn:\n", torch.randn(2, 3))

t3 = torch.arange(12).reshape(3, 4)
print("arange + reshape:\n", t3)
print("shape:", t3.shape)
print("转置 shape:", t3.T.shape)

a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])
print("a + b =", a + b)
print("a * b =", a * b, "(逐元素)")
print("a @ b =", a @ b, "(点积)")

A = torch.randn(3, 4)
B = torch.randn(4, 2)
C = A @ B
print(f"矩阵乘法: {list(A.shape)} @ {list(B.shape)} -> {list(C.shape)}")

从列表创建: tensor([1., 2., 3.]) dtype= torch.float32
从 NumPy 创建:
 tensor([[1., 2.],
        [3., 4.]])
zeros:
 tensor([[0., 0., 0.],
        [0., 0., 0.]])
ones:
 tensor([[1., 1., 1.],
        [1., 1., 1.]])
randn:
 tensor([[ 0.3367,  0.1288,  0.2345],
        [ 0.2303, -1.1229, -0.1863]])
arange + reshape:
 tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
shape: torch.Size([3, 4])
转置 shape: torch.Size([4, 3])
a + b = tensor([5., 7., 9.])
a * b = tensor([ 4., 10., 18.]) (逐元素)
a @ b = tensor(32.) (点积)
矩阵乘法: [3, 4] @ [4, 2] -> [3, 2]


## 2. 自动求导 Autograd

当 Tensor 设置 `requires_grad=True` 时，PyTorch 会记录它参与的计算，形成计算图。调用 `loss.backward()` 后，PyTorch 会沿计算图反向传播，把梯度写入每个叶子 Tensor 的 `.grad`。

初学者要记住三件事：

- `requires_grad=True` 表示需要梯度。
- `.backward()` 表示从当前标量结果开始反向传播。
- `.grad` 保存梯度。

In [3]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
y.backward()
print("y=x^2, x=3")
print("dy/dx 应为 6，PyTorch 计算:", x.grad)
assert x.grad == 6.0

x = torch.tensor(2.0, requires_grad=True)
y = (2 * x + 1) ** 3
y.backward()
print("y=(2x+1)^3, x=2")
print("dy/dx 应为 150，PyTorch 计算:", x.grad)
assert x.grad == 150.0

y=x^2, x=3
dy/dx 应为 6，PyTorch 计算: tensor(6.)
y=(2x+1)^3, x=2
dy/dx 应为 150，PyTorch 计算: tensor(150.)


In [4]:
x = torch.tensor(1.0, requires_grad=True)
w = torch.tensor(2.0, requires_grad=True)
bias = torch.tensor(0.5, requires_grad=True)

z = w * x + bias
a = torch.sigmoid(z)
loss = (a - 1.0) ** 2
loss.backward()

print(f"z = {z.item():.4f}")
print(f"a = sigmoid(z) = {a.item():.4f}")
print(f"loss = {loss.item():.6f}")
print("梯度:")
print("  dloss/dw =", round(w.grad.item(), 6))
print("  dloss/db =", round(bias.grad.item(), 6))
print("  dloss/dx =", round(x.grad.item(), 6))

z = 2.5000
a = sigmoid(z) = 0.9241
loss = 0.005754
梯度:
  dloss/dw = -0.010636
  dloss/db = -0.010636
  dloss/dx = -0.021272


## 3. nn.Module：构建神经网络

`nn.Module` 是 PyTorch 中所有网络模块的基类。标准写法：

1. 继承 `nn.Module`。
2. 在 `__init__` 中定义层。
3. 在 `forward` 中定义前向传播。

PyTorch 会自动跟踪 `nn.Linear` 等层里的参数，因此优化器可以通过 `model.parameters()` 找到所有需要更新的权重。

In [5]:
class SimpleNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.layer1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(hidden_size, output_size)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        x = self.sigmoid(x)
        return x


model = SimpleNet(input_size=2, hidden_size=16, output_size=1)
print(model)
print("总参数量:", sum(p.numel() for p in model.parameters()))
for name, param in model.named_parameters():
    print(f"{name}: shape={list(param.shape)}, 参数量={param.numel()}")

SimpleNet(
  (layer1): Linear(in_features=2, out_features=16, bias=True)
  (relu): ReLU()
  (layer2): Linear(in_features=16, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)
总参数量: 65
layer1.weight: shape=[16, 2], 参数量=32
layer1.bias: shape=[16], 参数量=16
layer2.weight: shape=[1, 16], 参数量=16
layer2.bias: shape=[1], 参数量=1


## 4. 完整训练流程

PyTorch 训练模板固定而重要：

```python
outputs = model(inputs)
loss = criterion(outputs, labels)
optimizer.zero_grad()
loss.backward()
optimizer.step()
```

`zero_grad()` 很重要，因为 PyTorch 默认会累加梯度。如果忘了清零，当前 batch 的梯度会和上一个 batch 混在一起。

In [6]:
X_moon, y_moon = make_moons(n_samples=1000, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X_moon, y_moon, test_size=0.2, random_state=42)

X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train).reshape(-1, 1)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.FloatTensor(y_test).reshape(-1, 1)

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

torch.manual_seed(42)
model = SimpleNet(input_size=2, hidden_size=32, output_size=1)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

print("--- 开始训练 ---")
for epoch in range(80):
    model.train()
    epoch_loss = 0
    for batch_X, batch_y in train_loader:
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    if epoch % 20 == 0 or epoch == 79:
        model.eval()
        with torch.no_grad():
            train_pred = (model(X_train_t) >= 0.5).float()
            test_pred = (model(X_test_t) >= 0.5).float()
            train_acc = (train_pred == y_train_t).float().mean()
            test_acc = (test_pred == y_test_t).float().mean()
        print(f"Epoch {epoch:3d}: loss={epoch_loss/len(train_loader):.4f}, train_acc={train_acc:.4f}, test_acc={test_acc:.4f}")

model.eval()
with torch.no_grad():
    y_pred_final = (model(X_test_t) >= 0.5).numpy().flatten()
final_acc = accuracy_score(y_test, y_pred_final)
print("最终测试准确率:", round(final_acc, 4))
assert final_acc > 0.85

--- 开始训练 ---
Epoch   0: loss=0.4760, train_acc=0.8500, test_acc=0.8300
Epoch  20: loss=0.0971, train_acc=0.9663, test_acc=0.9800
Epoch  40: loss=0.0771, train_acc=0.9725, test_acc=0.9750
Epoch  60: loss=0.0713, train_acc=0.9750, test_acc=0.9850
Epoch  79: loss=0.0695, train_acc=0.9725, test_acc=0.9900
最终测试准确率: 0.99


## 5. Sequential、Dropout、Weight Decay

对简单顺序网络，可以用 `nn.Sequential` 更简洁地写模型。

常用训练技巧：

- **Dropout**：训练时随机关闭一部分神经元，降低过拟合。
- **Weight Decay**：优化器中的 L2 正则化，限制权重过大。
- **model.train()/model.eval()**：切换训练和评估行为，Dropout 和 BatchNorm 会受影响。

In [7]:
torch.manual_seed(42)
model_seq = nn.Sequential(
    nn.Linear(2, 32),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(16, 1),
    nn.Sigmoid(),
)
print(model_seq)

optimizer_seq = optim.Adam(model_seq.parameters(), lr=0.01, weight_decay=1e-4)
criterion_seq = nn.BCELoss()

for epoch in range(80):
    model_seq.train()
    for batch_X, batch_y in train_loader:
        outputs = model_seq(batch_X)
        loss = criterion_seq(outputs, batch_y)
        optimizer_seq.zero_grad()
        loss.backward()
        optimizer_seq.step()

model_seq.eval()
with torch.no_grad():
    y_pred_seq = (model_seq(X_test_t) >= 0.5).numpy().flatten()
acc_seq = accuracy_score(y_test, y_pred_seq)
print("Sequential 模型测试准确率:", round(acc_seq, 4))
assert acc_seq > 0.85

Sequential(
  (0): Linear(in_features=2, out_features=32, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.2, inplace=False)
  (3): Linear(in_features=32, out_features=16, bias=True)
  (4): ReLU()
  (5): Dropout(p=0.2, inplace=False)
  (6): Linear(in_features=16, out_features=1, bias=True)
  (7): Sigmoid()
)
Sequential 模型测试准确率: 0.975


## 6. 保存和加载模型

推荐保存 `state_dict`，它只包含模型参数，不包含完整 Python 对象，更稳定也更常用。

加载时必须先创建相同结构的模型，再 `load_state_dict`。

In [8]:
save_path = ROOT_PATH = "/home/gfhong/testcode/ai_tutorial/04_deep_learning/model_demo.pth"
torch.save(model.state_dict(), save_path)
print("模型已保存:", save_path)

loaded_model = SimpleNet(input_size=2, hidden_size=32, output_size=1)
loaded_model.load_state_dict(torch.load(save_path, weights_only=True))
loaded_model.eval()

with torch.no_grad():
    y_pred_loaded = (loaded_model(X_test_t) >= 0.5).numpy().flatten()
assert np.array_equal(y_pred_final, y_pred_loaded)
print("✓ 加载的模型预测结果与原模型一致")

模型已保存: /home/gfhong/testcode/ai_tutorial/04_deep_learning/model_demo.pth
✓ 加载的模型预测结果与原模型一致


## 7. 总结与练习

关键要点：

1. Tensor = 多维数组 + GPU 支持 + 自动求导。
2. `requires_grad=True` 和 `.backward()` 是 Autograd 的核心。
3. `nn.Module` 用来组织模型层和参数。
4. 标准训练循环是 forward -> loss -> zero_grad -> backward -> step。
5. 推理时使用 `model.eval()` 和 `torch.no_grad()`。

**练习**：把隐藏层宽度、学习率、batch size 分别改一改，观察训练稳定性和测试准确率。

## 8. 深入理解：PyTorch 帮你省掉了什么

在上一节手写神经网络时，我们自己保存中间变量、手写梯度、更新参数。PyTorch 把这些工作拆成几个组件：

| 手写 NumPy | PyTorch 对应 |
|------------|--------------|
| 自己保存 $z_1,a_1,z_2$ | Autograd 自动记录计算图 |
| 自己推导 dW/db | `loss.backward()` 自动计算 |
| 自己写参数更新 | `optimizer.step()` |
| 自己组织层参数 | `nn.Module` |
| 自己分 batch | `DataLoader` |

但 PyTorch 不是魔法。你仍然需要理解训练循环：前向传播产生预测，损失衡量错误，反向传播计算梯度，优化器更新参数。

## 9. PyTorch 初学者高频错误

1. **忘记 `optimizer.zero_grad()`**
   PyTorch 默认累加梯度，不清零会导致梯度混入上一步结果。

2. **训练和评估模式混用**
   有 Dropout 或 BatchNorm 时，训练前用 `model.train()`，评估前用 `model.eval()`。

3. **推理时忘记 `torch.no_grad()`**
   推理不需要梯度，关闭梯度能节省内存并加快速度。

4. **输出层和损失函数搭配错误**
   二分类若用 `BCELoss`，模型输出通常要经过 Sigmoid；多分类若用 `CrossEntropyLoss`，模型输出 logits，不要手动 Softmax。

5. **Tensor shape 不匹配**
   这是最常见报错。训练前先打印输入、标签、输出形状，是非常好的习惯。

## 10. 实战训练模板

以后写 PyTorch 训练代码，可以从这个模板出发：

```python
for epoch in range(n_epochs):
    model.train()
    for batch_X, batch_y in train_loader:
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        # validation / test
        ...
```

这个模板是深度学习工程里的肌肉记忆。理解它，比记住某个具体网络更重要。